<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/torch/hard/optimizers/muon/muon.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("muon", ...)`

# Implement Muon from Scratch

**Difficulty**: 🔴 Hard

**Companies**: Meta, Google

---

### Problem Statement

Muon ("Momentum Orthogonalized by Newton–Schulz", Jordan et al., 2024) is the optimizer behind recent LLM training speedrun records:

- **Matrix parameters (`ndim >= 2`)** — momentum, then orthogonalize the update via Newton–Schulz, then apply it.
- **1D parameters** (biases, norm weights) — fall back to AdamW.

### Tasks

1. `newton_schulz(X, steps)` — approximate matrix orthogonalization (the helper Muon needs).
2. `MyMuon` — momentum followed by orthogonalization for matrix parameters; an AdamW fallback for 1D parameters.

### References

- Muon blog: https://kellerjordan.github.io/posts/muon/

In [ ]:
import math
import torch
import torch.nn as nn
from torch.optim import Optimizer


## Helper: Newton–Schulz Orthogonalization

Muon needs to turn an arbitrary update matrix into an approximately **orthogonal** one — same shape, but with `O @ O.T ≈ I` (for a wide matrix). SVD would do it exactly but is too slow inside a training loop; Muon uses the quintic **Newton–Schulz iteration** instead:

```
X_{k+1} = a·X_k + b·(X_k X_kᵀ) X_k + c·(X_k X_kᵀ)² X_k      a = 15/8,  b = −5/4,  c = 3/8
```

The iteration only converges when the spectral norm of the input is at most 1, so normalize first — the Frobenius norm is a cheap upper bound on the spectral norm.

**Tip:** for an `(m, n)` matrix, `X @ X.T` is `(m, m)` and `X.T @ X` is `(n, n)` — build whichever Gram matrix is *smaller*.


In [ ]:
def newton_schulz(X: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """
    Approximate orthogonalization of a matrix via Newton–Schulz iteration.

    Given X (m × n), return an approximately orthogonal matrix O of the same
    shape: O @ O.T ≈ I if m ≤ n, or O.T @ O ≈ I if m > n.

    Uses the quintic (5th-order) iteration with coefficients
    a = 15/8, b = -5/4, c = 3/8.

    Args:
        X:     input matrix, shape (m, n)
        steps: number of Newton–Schulz iterations (default 5)

    Returns:
        Orthogonalized matrix, same shape as X
    """
    # TODO: normalize X so its spectral norm is <= 1:
    #       X = X / (frobenius_norm(X) + 1e-10)
    # TODO: run `steps` quintic iterations. If m <= n work with A = X @ X.T:
    #           X = a*X + b*(A @ X) + c*(A @ A @ X)
    #       otherwise work with A = X.T @ X (the smaller Gram matrix):
    #           X = a*X + b*(X @ A) + c*(X @ A @ A)
    #       with a = 15/8, b = -5/4, c = 3/8
    ...


## Part 3: Muon

Muon ("Momentum Orthogonalized by Newton–Schulz") is a modern optimizer designed for training large transformers.

**Matrix parameters (`ndim >= 2`):**

1. Momentum: `buf ← μ·buf + g` — with Nesterov look-ahead, the update is `g + μ·buf`, otherwise just `buf`.
2. Orthogonalize the update with your `newton_schulz`.
3. Rescale by `√(max(m, n) / min(m, n))` — orthogonalization makes the update's magnitude depend on the matrix's aspect ratio, and this corrects for it.
4. Apply the same decoupled weight decay as AdamW, then `θ ← θ − lr·scale·update`.

**1D parameters** (biases, LayerNorm gains): orthogonalization is meaningless for a vector, so fall back to an **AdamW-style** update with its own betas and eps.

Typical LLM hyperparameters: `lr=0.02, momentum=0.95, nesterov=True, ns_steps=5, weight_decay=0.01`.


In [ ]:
class MyMuon(Optimizer):
    """
    Muon optimizer — Momentum with Newton–Schulz Orthogonalization.

    Args:
        params:       iterable of parameters to optimize
        lr:           learning rate (default 2e-2 — higher than Adam)
        momentum:     momentum coefficient μ (default 0.95)
        nesterov:     use Nesterov-style look-ahead momentum (default True)
        ns_steps:     Newton–Schulz iterations (default 5)
        weight_decay: decoupled weight decay (default 1e-2)
        adamw_betas:  betas for the AdamW fallback on 1D params (default (0.9, 0.95))
        adamw_eps:    epsilon for the AdamW fallback (default 1e-8)
    """

    def __init__(self, params, lr=2e-2, momentum=0.95, nesterov=True,
                 ns_steps=5, weight_decay=1e-2,
                 adamw_betas=(0.9, 0.95), adamw_eps=1e-8):
        defaults = dict(
            lr=lr, momentum=momentum, nesterov=nesterov,
            ns_steps=ns_steps, weight_decay=weight_decay,
            adamw_betas=adamw_betas, adamw_eps=adamw_eps,
        )
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            momentum = group['momentum']
            nesterov = group['nesterov']
            ns_steps = group['ns_steps']
            weight_decay = group['weight_decay']
            beta1, beta2 = group['adamw_betas']
            eps = group['adamw_eps']

            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad

                state = self.state[p]

                # State initialization
                if len(state) == 0:
                    state['step'] = 0
                    state['momentum_buffer'] = torch.zeros_like(p)
                    # AdamW state for the 1D fallback
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)

                state['step'] += 1
                buf: torch.Tensor = state['momentum_buffer']
                exp_avg: torch.Tensor = state['exp_avg']
                exp_avg_sq: torch.Tensor = state['exp_avg_sq']

                if p.ndim >= 2:
                    # ── Matrix parameter: Muon update ──

                    # TODO: momentum buffer:  buf = momentum * buf + grad
                    #       Nesterov look-ahead uses update = grad + momentum * buf,
                    #       plain momentum uses update = buf

                    # TODO: orthogonalize the update:
                    #       update = newton_schulz(update, steps=ns_steps)

                    # TODO: aspect-ratio scale: sqrt(max(m, n) / min(m, n))

                    # TODO: decoupled weight decay: p.mul_(1 - lr * weight_decay)

                    # TODO: apply the update: p -= lr * scale * update
                    ...
                else:
                    # ── 1D parameter (bias, norm gain): AdamW fallback ──

                    # TODO: biased moment estimates + bias corrections
                    #       (same as Adam/AdamW, with adamw_betas / adamw_eps)

                    # TODO: decoupled weight decay, then the AdamW update
                    ...

        return loss


## Validation

`newton_schulz` is checked for orthogonality, and Muon by training a tiny linear model.

Until your implementations are in place these tests will fail — that is expected.


In [ ]:
def test_newton_schulz():
    """Newton–Schulz should produce an approximately orthogonal matrix."""
    print("Testing newton_schulz...", end=" ")

    torch.manual_seed(42)
    m, n = 32, 64
    X = torch.randn(m, n)

    O = newton_schulz(X, steps=5)

    # For m <= n, O @ O.T should be close to I
    I_approx = O @ O.T
    diag_mean = I_approx.diag().mean().item()
    mask = ~torch.eye(m, dtype=torch.bool)
    offdiag_rms = I_approx[mask].square().mean().sqrt().item()

    if abs(diag_mean - 1.0) < 0.1 and offdiag_rms < 0.1:
        print(f"PASS  (diag mean: {diag_mean:.4f}, offdiag RMS: {offdiag_rms:.4f})")
    else:
        print(f"FAIL  (diag mean: {diag_mean:.4f}, offdiag RMS: {offdiag_rms:.4f})")




In [ ]:
def test_muon():
    """Muon should reduce the loss on a small linear model."""
    print("Testing Muon...", end=" ")

    torch.manual_seed(42)
    B, D_in, D_out = 32, 16, 8

    model = nn.Linear(D_in, D_out)
    opt = MyMuon(model.parameters(), lr=0.02, momentum=0.95, weight_decay=1e-3)

    X = torch.randn(B, D_in)
    y = torch.randn(B, D_out)

    initial_loss = ((model(X) - y) ** 2).mean().item()

    for _ in range(200):
        loss = ((model(X) - y) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()

    final_loss = ((model(X) - y) ** 2).mean().item()

    if final_loss < initial_loss * 0.5:
        print(f"PASS  (loss: {initial_loss:.4f} -> {final_loss:.4f})")
    else:
        print(f"FAIL  (loss: {initial_loss:.4f} -> {final_loss:.4f})")


test_newton_schulz()
test_adam()
test_adamw()
test_adamw_weight_decay()
test_muon()
